# 02 - Preprocessing

The cleaning below uses only properties of the data itself: how much of a column is
missing, how many distinct values it has, how far a value sits from the rest of its
column, and how strongly two columns move together. No knowledge of what the columns
measure is used at any point.

That is deliberate. The study asks whether a tabular foundation model still needs
conventional preprocessing, so the preprocessing has to be the conventional kind -
the rules a competent person applies to an unfamiliar table.

Two files come out of this notebook:

| file | what it is |
|---|---|
| `heart_disease_raw.csv` | all 920 records, missing values intact, target derived |
| `heart_disease_preprocessed.csv` | the fully cleaned table |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import *

apply_plot_style()
FIGURES_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Load and derive the target

In [ ]:
df = load_raw()
n0 = len(df)
print(f"{n0} records from {df[SITE_COL].nunique()} hospitals")

df[TARGET] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])
feats = feature_columns(df)

print(f"{int((df[TARGET]==0).sum())} negative / {int((df[TARGET]==1).sum())} positive")

`num` records disease severity from 0 to 4. It becomes a two-class target at
`num > 0`, and the original column is dropped - keeping it would hand the model the
answer.

The untouched version is saved now, before any rows or columns are removed. That is
the input the zero-shot model gets.

In [ ]:
df[["id", SITE_COL] + feats + [TARGET]].to_csv(
    PROCESSED_DIR / "heart_disease_raw.csv", index=False)
print(f"saved heart_disease_raw.csv  {len(df)} rows")

## Step 1 - drop mostly-empty columns

In [ ]:
miss = df[feats].isna().mean()
dropped = sorted(miss[miss > MISSING_COL_THRESHOLD].index)
feats = [c for c in feats if c not in dropped]
df = df.drop(columns=dropped)

for c in dropped:
    print(f"dropped {c:<6} {miss[c]*100:.1f}% missing")
print()
print((miss[feats] * 100).round(1).to_string())

Threshold is 50%. A column that is mostly absent carries little information,
and filling in that much of it would mean inventing most of the values.

## Step 2 - drop duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=feats + [TARGET], keep="first")
print(f"removed {before - len(df)} duplicate records, {len(df)} left")

## Step 3 - drop rows with any missing value

In [ ]:
before = len(df)
by_site_before = df[SITE_COL].value_counts()
df = df.dropna(subset=feats).reset_index(drop=True)
by_site_after = df[SITE_COL].value_counts()

print(f"removed {before - len(df)} of {before} "
      f"({(before-len(df))/before*100:.1f}%), {len(df)} left\n")
for s in sorted(by_site_before.index):
    print(f"  {s:<12} {by_site_before.get(s,0):>4} -> {by_site_after.get(s,0):>4}")

Every way of filling a gap assumes something about why the value is absent.
Dropping is the only option that invents nothing.

It is also the most expensive step, and the cost is very uneven - Cleveland loses
nothing while the other three sites lose most of their records.

## Step 4 - drop outliers

In [ ]:
cont = continuous_columns(df, feats)
keep = pd.Series(True, index=df.index)

for c in cont:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - IQR_MULTIPLIER * iqr, q3 + IQR_MULTIPLIER * iqr
    flagged = (df[c] < lo) | (df[c] > hi)
    print(f"  {c:<10} {int(flagged.sum()):>3} outside [{lo:.1f}, {hi:.1f}]")
    keep &= ~flagged

before = len(df)
by_site_before = df[SITE_COL].value_counts()
df = df[keep].reset_index(drop=True)
by_site_after = df[SITE_COL].value_counts()

print(f"\nremoved {before - len(df)} records, {len(df)} left\n")
for s in sorted(by_site_before.index):
    after = by_site_after.get(s, 0)
    flag = "   <-- entire hospital gone" if after == 0 else ""
    print(f"  {s:<12} {by_site_before.get(s,0):>4} -> {after:>4}{flag}")

Values far from the rest of their column are usually recording errors, and
they drag the mean and the scaling around. The rule is the textbook one: 1.5 x IQR
past the quartiles, applied to columns with more than 10 distinct values.

**Switzerland disappears here.** Its cholesterol values are all zero, which sits
below the lower fence, so every remaining Swiss record is flagged as an outlier and
removed. Nobody decided to exclude a hospital - a generic rule did it silently. This
is worth stating plainly in the report.

## Step 5 - drop constant and redundant columns

In [ ]:
const = [c for c in feats if df[c].nunique() <= 1]
if const:
    feats = [c for c in feats if c not in const]
    df = df.drop(columns=const)
print("constant columns dropped:", const or "none")

corr = df[feats].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
redundant = [c for c in upper.columns if (upper[c] > CORRELATION_THRESHOLD).any()]
if redundant:
    feats = [c for c in feats if c not in redundant]
    df = df.drop(columns=redundant)
print("redundant columns dropped:", redundant or "none")
print(f"\nstrongest remaining pair: {upper.max().max():.2f}")

## Step 6 - standardise the continuous columns

In [ ]:
cont = continuous_columns(df, feats)
df[cont] = (df[cont] - df[cont].mean()) / df[cont].std(ddof=0)
print("rescaled to mean 0, sd 1:", ", ".join(cont))
df[cont].describe().loc[["mean", "std"]].round(3)

These columns sit on very different numeric ranges, so anything that measures
distance or fits coefficients would be dominated by whichever column carries the
biggest numbers. The rest are left alone - few distinct values, so they look like
codes rather than measurements.

One caveat for the report: the mean and standard deviation are computed over the
whole file. When this is split in notebook 03, the test rows will already have fed
into those two numbers.

## Result

In [ ]:
out = df[["id", SITE_COL] + feats + [TARGET]]
out.to_csv(PROCESSED_DIR / "heart_disease_preprocessed.csv", index=False)

print(f"records:  {n0} -> {len(out)}  ({(n0-len(out))/n0*100:.1f}% removed)")
print(f"features: 13 -> {len(feats)}")
print(f"classes:  {int((out[TARGET]==0).sum())} negative / "
      f"{int((out[TARGET]==1).sum())} positive "
      f"({out[TARGET].mean()*100:.1f}% positive)")
print(f"hospitals left: {', '.join(out[SITE_COL].unique())}")
out.head()

## What the cleaning cost

In [ ]:
raw = pd.read_csv(PROCESSED_DIR / "heart_disease_raw.csv")
prep = out
sites = sorted(raw[SITE_COL].unique())

counts = [[int((raw[SITE_COL] == s).sum()) for s in sites],
          [int((prep[SITE_COL] == s).sum()) for s in sites]]
rates = [[frame.loc[frame[SITE_COL] == s, TARGET].mean()
          if (frame[SITE_COL] == s).any() else np.nan for s in sites]
         for frame in (raw, prep)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.6, 3.4))
x = np.arange(len(sites))
names = [f"Before  ({len(raw)} records)", f"After  ({len(prep)} records)"]

for k in (0, 1):
    bars = ax1.bar(x + (k - 0.5) * 0.4, counts[k], 0.36, color=CLASS_COLOURS[k],
                   edgecolor="white", linewidth=1, label=names[k])
    for b, v in zip(bars, counts[k]):
        ax1.text(b.get_x() + b.get_width() / 2, v + 6, str(v), ha="center",
                 fontsize=7.5, color="#52514e")
ax1.set_xticks(x, sites, fontsize=8.5)
ax1.set_ylabel("Records")
ax1.set_title("Records per hospital", fontsize=10, loc="left")
ax1.set_ylim(0, max(counts[0]) * 1.18)
ax1.xaxis.grid(False)
ax1.legend(fontsize=8)

offsets, label_dy = (-0.13, 0.13), (-15, 10)
for k in (0, 1):
    ok = ~np.isnan(rates[k])
    xs, ys = x[ok] + offsets[k], np.array(rates[k])[ok]
    ax2.scatter(xs, ys, s=70, color=CLASS_COLOURS[k], edgecolor="white",
                linewidth=1.2, zorder=3, label=names[k])
    for xi, v in zip(xs, ys):
        ax2.annotate(f"{v:.2f}", (xi, v), textcoords="offset points",
                     xytext=(0, label_dy[k]), ha="center", fontsize=7.5,
                     color="#52514e")
for xi in range(len(sites)):
    if np.isnan(rates[1][xi]):
        ax2.annotate("removed", (xi + offsets[1], rates[0][xi]),
                     textcoords="offset points", xytext=(4, 0), ha="left",
                     va="center", fontsize=7.5, color="#b52d2c")
ax2.set_xticks(x, sites, fontsize=8.5)
ax2.set_ylabel("Positive rate")
ax2.set_ylim(0, 1.12)
ax2.set_title("Disease prevalence per hospital", fontsize=10, loc="left")
ax2.xaxis.grid(False)
ax2.legend(fontsize=8, loc="lower right")

fig.suptitle("Effect of preprocessing on the sample", x=0.02, y=1.04, ha="left",
             fontsize=11)
fig.savefig(FIGURES_DIR / "eda_before_after.png")
plt.show()

Cleaning did not just shrink the sample, it changed which patients it
represents. One hospital is gone entirely, and prevalence shifted at every site that
survived. Whatever the models learn, they learn it from a population that no longer
matches the one the data was collected from.